# 🤖 EEG Classification — Left Hand vs Right Hand (Healthy Subjects)

**Input:** `features_lh_rh.csv` (output of `preprocessing.ipynb`)  
**Goal:** Binary classification LH (0) vs RH (1) using EEG features  

**Models:**
- Logistic Regression (baseline)
- Random Forest
- Support Vector Machine (RBF)
- XGBoost
- LightGBM

**Validation:** Stratified 5-Fold + Subject-wise Leave-One-Subject-Out (LOSO)  
**Saved outputs:** metrics CSV, confusion matrices, ROC curves, feature importances

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Sklearn
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, cohen_kappa_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report, roc_curve
)

# XGBoost & LightGBM (install if missing)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available. Install with: pip install xgboost')

try:
    from lightgbm import LGBMClassifier
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('LightGBM not available. Install with: pip install lightgbm')

import joblib
print('Libraries loaded.')

## 1. Configuration

In [ ]:
BASE_DIR    = '/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook'
INPUT_CSV   = os.path.join(BASE_DIR, 'features_lh_rh.csv')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
MODELS_DIR  = os.path.join(BASE_DIR, 'saved_models')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS      = 5
N_SELECT_K   = 50   # top-K features for SelectKBest

CLASS_NAMES  = ['LH', 'RH']
LABEL_COL    = 'label'

# Meta columns to exclude from features
META_COLS = {'subject_id', 'scenario_id', 'scenario', 'filename',
             'task', 'label', 'label_name'}

print('Config OK.')

## 2. Load Feature CSV

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f'Shape: {df.shape}')
print(f"Classes: {df['label_name'].value_counts().to_dict()}")

feature_cols = [c for c in df.columns if c not in META_COLS]
print(f'Feature columns: {len(feature_cols)}')

X = df[feature_cols].values.astype(float)
y = df[LABEL_COL].values
groups = df['subject_id'].values   # for LOSO

print(f'X shape: {X.shape},  y shape: {y.shape}')
print(f'Class balance — LH: {(y==0).sum()}, RH: {(y==1).sum()}')

## 3. Define Models

In [ ]:
def make_pipeline(clf, n_features=N_SELECT_K):
    """Standard pipeline: RobustScaler → SelectKBest → Classifier."""
    return Pipeline([
        ('scaler', RobustScaler()),
        ('select', SelectKBest(f_classif, k=min(n_features, X.shape[1]))),
        ('clf', clf)
    ])


MODELS = {
    'Logistic Regression': make_pipeline(
        LogisticRegression(C=1.0, max_iter=1000, random_state=RANDOM_STATE)
    ),
    'Random Forest': make_pipeline(
        RandomForestClassifier(n_estimators=200, max_depth=10,
                               random_state=RANDOM_STATE, n_jobs=-1)
    ),
    'SVM (RBF)': make_pipeline(
        SVC(kernel='rbf', C=1.0, gamma='scale',
            probability=True, random_state=RANDOM_STATE)
    ),
}

if HAS_XGB:
    MODELS['XGBoost'] = make_pipeline(
        XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                      use_label_encoder=False, eval_metric='logloss',
                      random_state=RANDOM_STATE, n_jobs=-1)
    )

if HAS_LGB:
    MODELS['LightGBM'] = make_pipeline(
        LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.05,
                       random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    )

print(f'Models defined: {list(MODELS.keys())}')

## 4. Stratified K-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

SCORING = ['accuracy', 'balanced_accuracy', 'f1', 'roc_auc']

cv_results = {}

for name, pipe in MODELS.items():
    print(f'\n→ {name} ...')
    cv = cross_validate(pipe, X, y, cv=skf, scoring=SCORING, n_jobs=-1,
                        return_train_score=False)
    cv_results[name] = cv
    print(f"   Acc  : {cv['test_accuracy'].mean():.4f} ± {cv['test_accuracy'].std():.4f}")
    print(f"   BAC  : {cv['test_balanced_accuracy'].mean():.4f} ± {cv['test_balanced_accuracy'].std():.4f}")
    print(f"   F1   : {cv['test_f1'].mean():.4f} ± {cv['test_f1'].std():.4f}")
    print(f"   AUC  : {cv['test_roc_auc'].mean():.4f} ± {cv['test_roc_auc'].std():.4f}")

print('\n✅ K-Fold CV done.')

## 5. CV Results Table

In [ ]:
rows = []
for name, cv in cv_results.items():
    rows.append({
        'Model'          : name,
        'Accuracy'       : f"{cv['test_accuracy'].mean():.4f} ± {cv['test_accuracy'].std():.4f}",
        'Bal. Accuracy'  : f"{cv['test_balanced_accuracy'].mean():.4f} ± {cv['test_balanced_accuracy'].std():.4f}",
        'F1'             : f"{cv['test_f1'].mean():.4f} ± {cv['test_f1'].std():.4f}",
        'ROC-AUC'        : f"{cv['test_roc_auc'].mean():.4f} ± {cv['test_roc_auc'].std():.4f}",
        'Acc_mean'       : cv['test_accuracy'].mean(),
        'BAC_mean'       : cv['test_balanced_accuracy'].mean(),
        'F1_mean'        : cv['test_f1'].mean(),
        'AUC_mean'       : cv['test_roc_auc'].mean(),
    })

results_df = pd.DataFrame(rows).sort_values('BAC_mean', ascending=False)
results_df.to_csv(os.path.join(RESULTS_DIR, 'cv_results.csv'), index=False)
print('CV results saved.')
results_df[['Model','Accuracy','Bal. Accuracy','F1','ROC-AUC']]

## 6. Model Comparison Bar Chart

In [ ]:
metrics = ['test_accuracy', 'test_balanced_accuracy', 'test_f1', 'test_roc_auc']
metric_labels = ['Accuracy', 'Balanced Acc', 'F1', 'ROC-AUC']
model_names = list(cv_results.keys())

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(model_names)))

for ax, metric, label in zip(axes, metrics, metric_labels):
    means = [cv_results[m][metric].mean() for m in model_names]
    stds  = [cv_results[m][metric].std()  for m in model_names]
    bars = ax.bar(model_names, means, yerr=stds, capsize=5, color=colors,
                  edgecolor='black', alpha=0.85)
    ax.set_ylim(0, 1.05)
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=30)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle(f'Stratified {N_FOLDS}-Fold CV — LH vs RH', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()

## 7. Leave-One-Subject-Out (LOSO) Validation

In [ ]:
# LOSO: train on all subjects except one, test on left-out subject
# This is the most realistic evaluation for EEG (cross-subject generalization)
logo = LeaveOneGroupOut()

loso_results = {}
best_model_name = results_df.iloc[0]['Model']  # best from CV

print(f'Running LOSO on best model: {best_model_name}')
print(f'Number of subjects (folds): {len(np.unique(groups))}')

loso_cv = cross_validate(
    MODELS[best_model_name], X, y,
    cv=logo, groups=groups,
    scoring=SCORING, n_jobs=-1
)

print(f"\nLOSO Results ({best_model_name}):")
for m, label in zip(['test_accuracy','test_balanced_accuracy','test_f1','test_roc_auc'],
                    ['Accuracy','BAC','F1','AUC']):
    print(f"  {label}: {loso_cv[m].mean():.4f} ± {loso_cv[m].std():.4f}")

loso_df = pd.DataFrame({
    'Subject': np.unique(groups)[:len(loso_cv['test_accuracy'])],
    'Accuracy': loso_cv['test_accuracy'],
    'BAC': loso_cv['test_balanced_accuracy'],
    'F1': loso_cv['test_f1'],
    'AUC': loso_cv['test_roc_auc'],
})
loso_df.to_csv(os.path.join(RESULTS_DIR, 'loso_results.csv'), index=False)
print('LOSO results saved.')

In [ ]:
# LOSO per-subject plot
fig, ax = plt.subplots(figsize=(max(10, len(loso_df)*0.4), 4))
x = np.arange(len(loso_df))
ax.bar(x - 0.2, loso_df['Accuracy'], 0.2, label='Accuracy', color='steelblue')
ax.bar(x,       loso_df['BAC'],      0.2, label='BAC',      color='coral')
ax.bar(x + 0.2, loso_df['AUC'],      0.2, label='AUC',      color='seagreen')
ax.axhline(loso_df['BAC'].mean(), color='coral', linestyle='--',
           alpha=0.7, label=f'Mean BAC={loso_df["BAC"].mean():.3f}')
ax.set_xticks(x)
ax.set_xticklabels(loso_df['Subject'], rotation=45, ha='right', fontsize=7)
ax.set_ylim(0, 1.1)
ax.set_title(f'LOSO per Subject — {best_model_name}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loso_per_subject.png'), dpi=150)
plt.show()

## 8. Detailed Evaluation — Best Model (Train/Test Split)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Fit best model
best_pipe = MODELS[best_model_name]
best_pipe.fit(X_train, y_train)

y_pred      = best_pipe.predict(X_test)
y_prob      = best_pipe.predict_proba(X_test)[:, 1]

print(f'\n=== {best_model_name} — Hold-out Test Set ===')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
print(f'Cohen Kappa : {cohen_kappa_score(y_test, y_pred):.4f}')
print(f'ROC-AUC     : {roc_auc_score(y_test, y_prob):.4f}')

## 9. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, normalize, title in zip(
    axes,
    [None, 'true'],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalized)']
):
    cm = confusion_matrix(y_test, y_pred, normalize=normalize)
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontweight='bold')

plt.suptitle(f'{best_model_name} — Hold-out Test', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

## 10. ROC Curves — All Models

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = plt.cm.tab10(np.linspace(0, 1, len(MODELS)))

for (name, pipe), color in zip(MODELS.items(), colors_roc):
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — LH vs RH', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=150)
plt.show()

## 11. Feature Importance (Best Model)

In [ ]:
# Re-fit best model to get selected features
best_pipe.fit(X_train, y_train)

selector   = best_pipe.named_steps['select']
clf_step   = best_pipe.named_steps['clf']
selected_mask   = selector.get_support()
selected_feats  = np.array(feature_cols)[selected_mask]

importances = None

if hasattr(clf_step, 'feature_importances_'):
    importances = clf_step.feature_importances_
    imp_label = 'Feature Importance'
elif hasattr(clf_step, 'coef_'):
    importances = np.abs(clf_step.coef_[0])
    imp_label = '|Coefficient|'

if importances is not None:
    imp_df = pd.DataFrame({'feature': selected_feats, 'importance': importances})
    imp_df = imp_df.sort_values('importance', ascending=False)
    imp_df.to_csv(os.path.join(RESULTS_DIR, 'feature_importance.csv'), index=False)

    top_n = min(30, len(imp_df))
    top_imp = imp_df.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top_imp['feature'][::-1], top_imp['importance'][::-1],
            color='steelblue', edgecolor='black', alpha=0.85)
    ax.set_xlabel(imp_label)
    ax.set_title(f'Top {top_n} Features — {best_model_name}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance.png'), dpi=150)
    plt.show()
    print(f'Feature importance saved. Top 5:')
    print(imp_df.head().to_string(index=False))
else:
    print('Feature importance not available for this model type.')

## 12. Subband × Channel Feature Importance Heatmap

In [ ]:
if importances is not None:
    # Parse feature names: channel_subband_featuretype
    def parse_feat(name):
        parts = name.split('_')
        if len(parts) >= 2:
            return parts[0], parts[1]  # channel, subband
        return name, 'Unknown'

    imp_df['channel_'] = imp_df['feature'].apply(lambda x: parse_feat(x)[0])
    imp_df['subband_'] = imp_df['feature'].apply(lambda x: parse_feat(x)[1])

    pivot_imp = imp_df.groupby(['channel_', 'subband_'])['importance'].sum().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot_imp, annot=True, fmt='.3f', cmap='YlOrRd',
                linewidths=0.5, ax=ax)
    ax.set_title('Total Feature Importance: Channel × Subband', fontweight='bold')
    ax.set_xlabel('Subband')
    ax.set_ylabel('Channel')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'importance_heatmap.png'), dpi=150)
    plt.show()
else:
    print('Skipping heatmap (no feature importances).')

## 13. Final Metrics Summary

In [ ]:
# Collect all model metrics on hold-out set
all_metrics = []
for name, pipe in MODELS.items():
    pipe.fit(X_train, y_train)
    yp   = pipe.predict(X_test)
    yprob = pipe.predict_proba(X_test)[:, 1]
    all_metrics.append({
        'Model'          : name,
        'Accuracy'       : accuracy_score(y_test, yp),
        'Balanced_Acc'   : balanced_accuracy_score(y_test, yp),
        'F1'             : f1_score(y_test, yp),
        'ROC_AUC'        : roc_auc_score(y_test, yprob),
        'Cohen_Kappa'    : cohen_kappa_score(y_test, yp),
        'CV_BAC_mean'    : results_df[results_df['Model']==name]['BAC_mean'].values[0],
        'CV_AUC_mean'    : results_df[results_df['Model']==name]['AUC_mean'].values[0],
    })

final_df = pd.DataFrame(all_metrics).sort_values('Balanced_Acc', ascending=False)
final_df.to_csv(os.path.join(RESULTS_DIR, 'final_metrics.csv'), index=False)
print('Final metrics saved.')
final_df.set_index('Model').round(4)

In [ ]:
# Final summary heatmap
plot_cols = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'Cohen_Kappa']
hm_data   = final_df.set_index('Model')[plot_cols]

fig, ax = plt.subplots(figsize=(9, max(4, len(MODELS)*0.8)))
sns.heatmap(hm_data, annot=True, fmt='.4f', cmap='RdYlGn',
            vmin=0.3, vmax=1.0, linewidths=0.5, ax=ax)
ax.set_title('All Models — Hold-out Test Metrics', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'metrics_heatmap.png'), dpi=150)
plt.show()

## 14. Save Best Model

In [ ]:
# Fit best model on full data and save
best_pipe_final = MODELS[best_model_name]
best_pipe_final.fit(X, y)

model_path = os.path.join(MODELS_DIR, f'{best_model_name.replace(" ", "_")}_best.pkl')
joblib.dump(best_pipe_final, model_path)
print(f'✅ Best model saved: {model_path}')

# Save feature list for inference
feat_path = os.path.join(MODELS_DIR, 'feature_cols.txt')
with open(feat_path, 'w') as f:
    f.write('\n'.join(feature_cols))
print(f'✅ Feature list saved: {feat_path}')

print('\n=== All outputs saved to:', RESULTS_DIR, '===')
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size  = os.path.getsize(fpath)
    print(f'  {fname:<40} {size:>8,} bytes')